# The Anatomy of AI · Lecture 17 · The Prism — the notebook

In the lecture we built a **prism**: a machine that takes a few mixed numbers, spreads them over many *units* of which
only a few may be on, and rebuilds the numbers from those units. (Its usual name is a *sparse autoencoder*.) Here you
build it yourself:

1. on Lecture 15's toy, where five ideas hide in two neurons, and see a penalty make the prism find the five ideas;
2. inside InceptionV1, where a prism for one branch of layer **mixed3b** finds curve detectors in the gaps between
   Lecture 10's ten, and splits neuron **3b:359**, which has two jobs, into one unit per job.

Every key number is checked against the lecture's own: a ✓ means you got what the lecture got. The toy half runs on the
CPU in about two minutes; the InceptionV1 half wants a GPU (*Runtime → Change runtime type → T4 GPU*). The whole notebook
takes about ten minutes on free Colab.

In [ ]:
# Pinned installs. Free Colab already has torch and torchvision; a machine without them gets the pinned pair.
import importlib.util, subprocess, sys
def pip(*args):                                   # quiet: pip's output is shown only if an install fails
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", *args],
                       capture_output=True, text=True)
    if r.returncode:
        raise RuntimeError(r.stdout[-3000:] + r.stderr[-3000:])
if importlib.util.find_spec("torch") is None:
    pip("torch==2.5.1", "torchvision==0.20.1")
pip("numpy==2.0.2", "matplotlib==3.10.0", "pillow==11.1.0", "pyarrow==18.1.0", "requests==2.32.3")
pip("--no-deps", "torch-lucent==0.1.8")       # only its InceptionV1 port is used

In [ ]:
import io, json, math, os, time, warnings
warnings.filterwarnings("ignore", category=FutureWarning)      # library notices, not results
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEV)
T0 = time.time()

# The course's look: black ink on paper, no colour except real photos.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
    "lines.color": INK, "patch.edgecolor": INK, "axes.prop_cycle": plt.cycler(color=[INK, INK2]),
})


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "\u2212")


def check(name, value, expected, tol=0):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    extra = f" ± {tol:g}" if isinstance(expected, (int, float)) and not isinstance(expected, bool) and tol else ""
    print(f"{'✓' if ok else '✗'} {name}: {minus(shown)}   (lecture: {minus(expected)}{extra})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")


def fetch(url, path, what):
    """Download a file once (quietly), and say in one line what came down."""
    import requests
    if not os.path.exists(path):
        t = time.time()
        with requests.get(url, stream=True, timeout=120) as r:
            r.raise_for_status()
            with open(path + ".part", "wb") as f:
                for chunk in r.iter_content(1 << 22):
                    f.write(chunk)
        os.rename(path + ".part", path)
        print(f"downloaded {what} ({os.path.getsize(path) / 1e6:.1f} MB) in {time.time() - t:.0f} s")
    return path

## Part 1 · The prism on a toy

### 1. Five ideas, two neurons (Plate I)
Lecture 15 trained a tiny network to squeeze five *ideas* (numbers that are usually 0, and sometimes anything between 0
and 1) into two neurons, and to get them back out. Its weights `W` (2 × 5) put each idea on an arrow in the plane of
the two neurons: a pentagon. These are Lecture 15's own trained weights (five equal ideas, each absent 90% of the time,
seed 0).

In [ ]:
torch.set_num_threads(2)                       # the toy is tiny: the CPU is quicker than a GPU for it
N, M, S = 5, 2, 0.9                            # five ideas, two neurons, each idea absent 90% of the time
W = torch.tensor([[-0.26601099967956543, 0.9333760142326355, -0.41075399518013, 0.8442729711532593, -1.0968719720840454], [1.0671720504760742, 0.5838810205459595, -1.0198190212249756, -0.7068930268287659, 0.07611799985170364]], dtype=torch.float32)   # Lecture 15's trained weights: column i is idea i's arrow
b15 = torch.tensor([-0.20937299728393555, -0.2104559987783432, -0.21003399789333344, -0.20946499705314636, -0.20926600694656372], dtype=torch.float32)

ang = sorted(math.degrees(math.atan2(W[1, i], W[0, i])) % 360 for i in range(N))
print("arrow directions (degrees):", [round(a, 1) for a in ang])
print("arrow lengths:", [round(float(v), 2) for v in W.norm(dim=0)])
for a, e in zip(ang, [32.0, 104.0, 176.0, 248.1, 320.1]):
    check(f"an idea's direction", round(a, 1), e, tol=0.05)

fig, ax = plt.subplots(figsize=(3.6, 3.6))
for i in range(N):
    ax.annotate("", xy=(W[0, i], W[1, i]), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.6))
    ax.text(1.18 * W[0, i], 1.18 * W[1, i], str(i + 1), ha="center", va="center")
ax.axhline(0, color=INK2, lw=0.6); ax.axvline(0, color=INK2, lw=0.6)
ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4); ax.set_aspect("equal")
ax.set_xlabel("neuron 1"); ax.set_ylabel("neuron 2"); ax.set_title("Five ideas in two neurons", fontsize=10)
plt.show()

### 2. A machine that copies (Plate III)
An **autoencoder** copies its input through a narrow middle. Lecture 15's network is one: the *encoder* squeezes five
ideas into two numbers, `h = W x`; the *decoder* rebuilds five from two with the same weights turned round, a bias and
Lecture 1's gate, `x' = ReLU(Wᵀ h + b)`. Its **rebuild error** is how far `x'` is from `x` (summed over the five, averaged
over many inputs).

In [ ]:
def sample(n, seed):
    # n inputs with five ideas each: 0 with probability S, otherwise uniform between 0 and 1
    g = torch.Generator().manual_seed(seed)
    x = torch.rand(n, N, generator=g) * (torch.rand(n, N, generator=g) > S)
    return x, x @ W.T                          # the ideas, and what the two neurons hold

x, _ = sample(200000, seed=7)
x_rebuilt = torch.relu(x @ W.T @ W + b15)
err15 = ((x - x_rebuilt) ** 2).sum(1).mean().item()
print(f"Lecture 15's copier: rebuild error {err15:.4f}")
check("the copier's rebuild error (Lecture 15's final loss)", err15, 0.0264, tol=0.0015)

### 3. Turn it inside out (Plates IV–V)
A **prism** goes the other way: from the two numbers *out* to more units (here five), then back to two. Each unit is a
neuron of Lecture 1: weights, a bias, the gate. Each unit also owns one arrow, its entry in the prism's **dictionary**;
to rebuild, the prism adds up its units' arrows, each scaled by how strongly the unit is on.

    f = ReLU(W_e (h − b_d) + b_e)        the units (5 numbers, most of them 0)
    h' = W_d f + b_d                     the rebuilt two numbers (column i of W_d is unit i's arrow)

The prism only ever sees `h`, the two numbers. It is never told the ideas, nor `W`. We train it exactly as the lecture
did (20,000 steps of 4,096 inputs, Adam), on 200,000 samples, and test on 50,000 fresh ones. Five seeds train side by
side, each with its own random numbers, so you get exactly the lecture's five prisms.

In [ ]:
import itertools

def train_prisms(h, n_units, seeds, lam=0.0, topk=None, steps=20000, lr=3e-3):
    '''Train one prism per seed, all at once (each seed keeps its own random numbers, as if trained alone):
    20,000 steps of 4,096 inputs, Adam, the learning rate eased down to zero.'''
    S_ = len(seeds)
    gens = [torch.Generator().manual_seed(s) for s in seeds]
    Wd0 = []
    for g in gens:
        w = torch.randn(M, n_units, generator=g)
        Wd0.append(w / w.norm(dim=0, keepdim=True))
    Wd = torch.nn.Parameter(torch.stack(Wd0))                         # seeds x 2 x units: one arrow per unit
    We = torch.nn.Parameter(Wd.detach().transpose(1, 2).clone())       # seeds x units x 2
    be = torch.nn.Parameter(torch.zeros(S_, 1, n_units))
    bd = torch.nn.Parameter(torch.zeros(S_, 1, M))
    opt = torch.optim.Adam([Wd, We, be, bd], lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    for t in range(steps):
        hb = torch.stack([h[torch.randint(0, h.shape[0], (4096,), generator=g)] for g in gens])   # seeds x 4096 x 2
        f = torch.relu((hb - bd) @ We.transpose(1, 2) + be)
        if topk:                                                   # keep only the k strongest units (Plate VIII)
            v, i = f.topk(topk, dim=2)
            f = torch.zeros_like(f).scatter(2, i, v)
        hh = f @ Wd.transpose(1, 2) + bd
        loss = (((hb - hh) ** 2).sum(2).mean(1) + lam * (f * Wd.norm(dim=1, keepdim=True)).sum(2).mean(1)).sum()
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return [dict(Wd=Wd[k].detach(), We=We[k].detach(), be=be[k, 0].detach(), bd=bd[k, 0].detach(), topk=topk)
            for k in range(S_)]


def units(p, h):
    f = torch.relu((h - p["bd"]) @ p["We"].T + p["be"])
    if p["topk"]:
        v, i = f.topk(p["topk"], dim=1)
        f = torch.zeros_like(f).scatter(1, i, v)
    return f


def lineup(p):
    '''How well the dictionary's arrows line up with the five ideas: pair each idea with one arrow (the best pairing)
    and average the cosines (1 = every arrow points exactly along an idea).'''
    d = (p["Wd"] / p["Wd"].norm(dim=0, keepdim=True)).numpy(); t = (W / W.norm(dim=0, keepdim=True)).numpy()
    cos = t.T @ d
    best = max(itertools.permutations(range(cos.shape[1]), N), key=lambda pm: sum(cos[i, pm[i]] for i in range(N)))
    return float(np.mean([cos[i, best[i]] for i in range(N)])), list(best)


def lost(p, h):
    '''Variance lost: the rebuild error as a share of how much h varies.'''
    hh = units(p, h) @ p["Wd"].T + p["bd"]
    return float(((h - hh) ** 2).sum() / ((h - h.mean(0)) ** 2).sum())


x_train, h_train = sample(200000, seed=1)
x_test, h_test = sample(50000, seed=2)
t = time.time()
plain = train_prisms(h_train, 5, seeds=range(5), lam=0.0)
print(f"five prisms with no penalty trained in {time.time() - t:.0f} s")

**Too easy.** With five units for two numbers, copying is easy, so the copy is perfect. But nothing tells the arrows
where to point.

In [ ]:
rng = np.random.default_rng(2)                 # the lecture's one-idea test inputs (numpy, seed 2)
xs = rng.random((50000, N)) * (rng.random((50000, N)) > S)
present = xs > 0; single = present.sum(1) == 1; some = present.any(1)
hs = torch.tensor(xs @ W.numpy().T, dtype=torch.float32)


def naming(p):
    '''For inputs holding exactly one idea: is the strongest unit the one paired with that idea? (Plate V)'''
    _, pair = lineup(p)
    f = units(p, hs).numpy()[:, pair]
    return float((f[single].argmax(1) == present[single].argmax(1)).mean())


for s, p in enumerate(plain):
    c, _ = lineup(p)
    print(f"seed {s}: variance lost {100 * lost(p, h_test):.1f}%, line-up {c:.3f}, strongest unit names the idea {100 * naming(p):.0f}%")
    check(f"no penalty, seed {s}: line-up", c, [0.857, 0.934, 0.771, 0.918, 0.932][s], tol=0.03)
check("no penalty: variance lost, worst seed", max(lost(p, h_test) for p in plain), 0.0, tol=1e-4)
check("no penalty: strongest unit names the idea (median of 5 seeds)", float(np.median([naming(p) for p in plain])), 0.216, tol=0.03)

### 4. Charge for being on (Plates VI–VII)
The fix: every unit that is on costs something. The loss becomes

    rebuild error + penalty × Σᵢ fᵢ · |arrowᵢ|

(the *L1 penalty*, each unit weighted by its arrow's length, so the prism can't cheat by shrinking units and growing
arrows). A point that holds one idea is now cheapest to rebuild with one unit whose arrow points right at it.

In [ ]:
t = time.time()
charged = train_prisms(h_train, 5, seeds=range(5), lam=0.3)
print(f"five prisms with penalty 0.3 trained in {time.time() - t:.0f} s")


def exact10(p):
    '''For inputs with any idea: are the units above 10% of the strongest exactly the ideas present?'''
    _, pair = lineup(p)
    f = units(p, hs).numpy()[:, pair]
    on = f > 0.1 * f.max(1, keepdims=True)
    return float((on[some] == present[some]).all(1).mean())


def length_ratio(p):
    '''Shrinkage: for one-idea inputs, the rebuilt arrow's length over the true one (median).'''
    hh = (units(p, hs) @ p["Wd"].T + p["bd"]).numpy()
    return float(np.median(np.linalg.norm(hh, axis=1)[single] / np.linalg.norm(hs.numpy(), axis=1)[single]))


for s, p in enumerate(charged):
    c, _ = lineup(p)
    print(f"seed {s}: variance lost {100 * lost(p, h_test):.1f}%, line-up {c:.3f}, names exactly the ideas present "
          f"{100 * exact10(p):.0f}%, a one-idea input rebuilt at {100 * length_ratio(p):.0f}% of its length")
    check(f"penalty 0.3, seed {s}: line-up", c, [1.0, 1.0, 1.0, 1.0, 1.0][s], tol=0.002)
check("penalty 0.3: variance lost (median)", float(np.median([lost(p, h_test) for p in charged])), 0.0972, tol=0.003)
check("penalty 0.3: names exactly the ideas present (median)", float(np.median([exact10(p) for p in charged])), 0.67, tol=0.01)
check("penalty 0.3: strongest unit names the idea (median)", float(np.median([naming(p) for p in charged])), 0.803, tol=0.01)
check("penalty 0.3: shrinkage, rebuilt length (median)", float(np.median([length_ratio(p) for p in charged])), 0.554, tol=0.01)

fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.6))
for ax, p, title in ((axs[0], plain[0], "no penalty (seed 0)"), (axs[1], charged[0], "penalty 0.3 (seed 0)")):
    for i in range(N):
        ax.plot([0, W[0, i]], [0, W[1, i]], ls="--", color=INK2, lw=1)
    d = p["Wd"] / p["Wd"].norm(dim=0, keepdim=True) * 1.1
    for j in range(5):
        ax.annotate("", xy=(d[0, j], d[1, j]), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=INK, lw=1.8))
    ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4); ax.set_aspect("equal"); ax.set_title(title, fontsize=10)
fig.suptitle("Dashed: the five ideas. Solid: the prism's dictionary arrows.", fontsize=10)
plt.show()

### 5. What it was never told (Plate IX)
For inputs holding one idea, average each unit's strength. With the penalty, each idea has its own unit: the prism found
the five ideas from the two numbers alone.

In [ ]:
_, pair = lineup(charged[0])
f = units(charged[0], hs).numpy()[:, pair]
told = np.stack([f[single & present[:, i]].mean(0) for i in range(N)])
fig, ax = plt.subplots(figsize=(3.8, 3.4))
ax.imshow(told / told.max(), cmap="Greys", vmin=0, vmax=1)
ax.set_xticks(range(N), [f"unit {j + 1}" for j in range(N)], fontsize=8); ax.set_yticks(range(N), [f"idea {i + 1}" for i in range(N)], fontsize=8)
ax.set_title("penalty 0.3: which unit answers which idea", fontsize=10); plt.show()
diag = float(np.mean([told[i, i] / told[i].sum() for i in range(N)]))
print(f"on average {100 * diag:.1f}% of a one-idea input's unit activity is on its own unit")
check("each idea has its own unit (the strongest unit is on the diagonal for all five)", bool((told.argmax(1) == np.arange(N)).all()), True)

## Part 2 · The prism on InceptionV1

### 6. Back to InceptionV1 (Plate XII)
Layer **mixed3b** looks at a photo in 28 × 28 places and gives 480 numbers at each. Lecture 10's curve detectors live
in one branch of it, the *5 × 5 branch*: numbers 320 to 415, 96 in all. We load the network (with its normalization
fixed, and the fix checked) and the course's photos: photo 24,424 of ImageNet's validation set is a wall clock, photo
17,205 a kitchen scale.

In [ ]:
# The photos: ImageNet's 50,000 validation photos, in the exact copy the course measured (a public 256-pixel copy,
# benjamin-paine/imagenet-1k-256x256 on Hugging Face: parquet files of 25,000 photos, about 350 MB each, no key
# needed). Photo n here is photo n in the lecture. We read 100 photos at a time, only the ones we ask for.
import pyarrow.parquet as pq
from PIL import Image

SHARD = ("https://huggingface.co/datasets/benjamin-paine/imagenet-1k-256x256/resolve/main/data/"
         "validation-{:05d}-of-00002.parquet")
_files = {}


def _shard(s):
    path = fetch(SHARD.format(s), f"val-{s}.parquet", f"photo file {s}")
    if s not in _files:
        _files[s] = pq.ParquetFile(path)
    return _files[s]


def rows(ids):
    """{n: (jpeg bytes, class number 0..999)} for the photos with these numbers."""
    groups = {}
    for n in ids:
        groups.setdefault((n // 25000, (n % 25000) // 100), []).append(int(n))
    out = {}
    for (s, g), ns in sorted(groups.items()):
        t = _shard(s).read_row_group(g, columns=["image", "label"])
        ims, labs = t.column("image").to_pylist(), t.column("label").to_pylist()
        for n in ns:
            out[n] = (ims[n % 100]["bytes"], int(labs[n % 100]))
    return out


def decode(b, size=224):
    """JPEG bytes -> the whole (square) photo shrunk to 224 x 224, as a 3 x 224 x 224 tensor in [0, 1].
    (This is how Lecture 17 fed its photos to the network.)"""
    im = Image.open(io.BytesIO(b)).convert("RGB")
    w, h = im.size
    s = min(w, h)
    im = im.crop(((w - s) // 2, (h - s) // 2, (w - s) // 2 + s, (h - s) // 2 + s)).resize((size, size), Image.BILINEAR)
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0).permute(2, 0, 1)

In [ ]:
# InceptionV1 (GoogLeNet, 2014), the network Lecture 17 opens, through lucent's PyTorch port.
# One fix first: the port's local response normalization used the wrong constants (lucent pull request #51, Gorton 2024).
# We correct the installed file before importing it, as every lecture in the course does.
import importlib, importlib.util, pathlib, contextlib
_spec = importlib.util.find_spec("lucent")
_f = pathlib.Path(_spec.origin).parent / "modelzoo" / "inceptionv1" / "InceptionV1.py"
_s = _f.read_text()
_old = "size=9, alpha=9.99999974738e-05, beta=0.5, k=1"
if _old in _s:
    _f.write_text(_s.replace(_old, "size=10, alpha=10 * 0.00009999999747378752, beta=0.5, k=2"))
for _p in (_f.parent / "__pycache__").glob("InceptionV1*.pyc"):
    _p.unlink()                                   # a stale cached copy would keep the old constants
importlib.invalidate_caches()

from lucent.modelzoo import inceptionv1
with contextlib.redirect_stderr(io.StringIO()):            # keep the download's percentage bar off the page
    net = inceptionv1(pretrained=True).eval().to(DEV)
print("InceptionV1's weights (inception5h, 27 MB): ready")
for p in net.parameters():
    p.requires_grad_(False)

# Make sure the fix took: watch the normalization during one pass.
_M = importlib.import_module("lucent.modelzoo.inceptionv1.InceptionV1")
_seen, _orig = [], _M.F.local_response_norm
_M.F.local_response_norm = lambda x, size, alpha=1e-4, beta=0.75, k=1.0: (_seen.append((size, k)), _orig(x, size, alpha, beta, k))[1]
with torch.no_grad():
    net(torch.zeros(1, 3, 224, 224, device=DEV))
_M.F.local_response_norm = _orig
check("normalization size and k, every call", sorted(set(_seen)), [(10, 2)])


def prep(x01):
    """The port takes RGB in [-117, 138], as the original TensorFlow model did."""
    return x01 * 255.0 - 117.0


class Taps:
    """Look inside the network. record=[module names] keeps their outputs in taps[name]."""
    def __init__(self, record=()):
        mods = dict(net.named_modules())
        self.out, self.h = {}, []
        for name in record:
            self.h.append(mods[name].register_forward_hook(lambda m, i, o, n=name: self.out.__setitem__(n, o)))

    def __getitem__(self, name):
        return self.out[name]

    def __enter__(self):
        return self

    def __exit__(self, *a):
        for h in self.h:
            h.remove()


from torchvision.models import GoogLeNet_Weights
NAMES = GoogLeNet_Weights.IMAGENET1K_V1.meta["categories"]       # the 1,000 ImageNet class names, standard order

In [ ]:
R = rows([24424, 17205])
photos = torch.stack([decode(R[n][0]) for n in (24424, 17205)])
print("photo 24424:", NAMES[R[24424][1]], "· photo 17205:", NAMES[R[17205][1]])
check("photo 24424's class", R[24424][1], 409)
check("photo 17205's class", R[17205][1], 778)
with torch.no_grad(), Taps(["mixed3b"]) as T:
    net(prep(photos.to(DEV)))
    B5 = T["mixed3b"][:, 320:416].float().cpu()            # the 5 x 5 branch: 2 photos x 96 x 28 x 28
clock = B5[0, :, 3, 14]                                     # the place on the clock's rim the lecture follows
print(f"clock, place (3, 14): {(clock > 0).sum().item()} of 96 numbers above zero; the tallest is neuron "
      f"3b:{320 + int(clock.argmax())} = {clock.max():.0f}")
check("clock: numbers above zero at (3, 14)", int((clock > 0).sum()), 21)
check("clock: neuron 3b:359 at (3, 14)", clock[39].item(), 1010, tol=1.5)

fig, axs = plt.subplots(1, 2, figsize=(9, 3.2), gridspec_kw=dict(width_ratios=[1, 2.4]))
axs[0].imshow(photos[0].permute(1, 2, 0)); axs[0].axis("off")
axs[0].add_patch(plt.Rectangle((14 * 8, 3 * 8), 8, 8, fill=False, ec="white", lw=2))
axs[0].set_title("photo 24424, place (3, 14)", fontsize=9)
axs[1].bar(range(320, 416), clock, color=INK, width=0.8); axs[1].set_xlabel("neuron of the 5 x 5 branch")
axs[1].set_title("the 96 numbers at that place", fontsize=9); plt.show()

### 7. Build a prism for the branch yourself (Plate XIII)
Following Gorton (2024), a prism for the 96 numbers: **768 units**, eight per neuron, L1 penalty 1 (each unit weighted
by its arrow's length), inputs scaled so their average squared length is 96. The lecture trained on 1,281,200 places
from 160,150 *training* photos for 40,000 steps. We do the same recipe, smaller: 16 random places in each of 4,000
validation photos, 6,000 steps. Then we test it on places from 500 other photos.

In [ ]:
def branch_vectors(ids, per_photo, seed):
    g = torch.Generator().manual_seed(seed)
    out = []
    with torch.no_grad(), Taps(["mixed3b"]) as T:
        for s in range(0, len(ids), 100):
            chunk = ids[s:s + 100]
            r = rows(chunk)
            x = torch.stack([decode(r[n][0]) for n in chunk]).to(DEV)
            net(prep(x))
            a = T["mixed3b"][:, 320:416].flatten(2).transpose(1, 2)      # photos x 784 places x 96
            pick = torch.randint(0, 784, (len(chunk), per_photo), generator=g).to(DEV)
            out.append(torch.gather(a, 1, pick[..., None].expand(-1, -1, 96)).reshape(-1, 96).float().cpu())
    return torch.cat(out)


t = time.time()
X = branch_vectors(list(range(4000)), 16, seed=0)
Xt = branch_vectors(list(range(4000, 4500)), 16, seed=1)
print(f"{len(X):,} training places and {len(Xt):,} test places in {time.time() - t:.0f} s")


def train_branch_prism(X, n=768, lam=1.0, steps=6000, batch=4096, lr=1e-3, seed=0):
    d = X.shape[1]
    scale = float((d / (X ** 2).sum(1).mean()).sqrt())             # so that an input's mean squared length is d
    Xs = X.to(DEV) * scale
    g = torch.Generator().manual_seed(seed)
    Wd = torch.randn(d, n, generator=g); Wd = Wd / Wd.norm(dim=0, keepdim=True) * 0.1
    P = {k: v.to(DEV).requires_grad_(True) for k, v in dict(Wd=Wd.clone(), We=Wd.T.clone() / (Wd.norm(dim=0)[:, None] ** 2) * 0.1,
                                                              be=torch.zeros(n), bd=torch.zeros(d)).items()}
    opt = torch.optim.Adam(list(P.values()), lr=lr)
    warm = steps // 20
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda t: min(1.0, (t + 1) / warm) * (1.0 if t < 0.8 * steps else (steps - t) / (0.2 * steps)))
    gi = torch.Generator(device=DEV).manual_seed(seed)
    for t in range(steps):
        x = Xs[torch.randint(0, Xs.shape[0], (batch,), device=DEV, generator=gi)]
        f = torch.relu((x - P["bd"]) @ P["We"].T + P["be"])
        loss = ((x - (f @ P["Wd"].T + P["bd"])) ** 2).sum(1).mean() + lam * min(1.0, t / warm) * (f * P["Wd"].norm(dim=0)).sum(1).mean()
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
    return {k: v.detach() for k, v in P.items()} | {"scale": scale}


def encode(p, a):
    '''The prism's units for some branch numbers a (... x 96), in the network's own units.'''
    a = torch.as_tensor(a, dtype=torch.float32, device=p["We"].device)
    return torch.relu((a * p["scale"] - p["bd"]) @ p["We"].T + p["be"]) / p["scale"]


def rebuild(p, f):
    return ((f * p["scale"]) @ p["Wd"].T + p["bd"]) / p["scale"]


def fit(p, X):
    f = encode(p, X)
    Xd = X.to(f.device)
    return float(((Xd - rebuild(p, f)) ** 2).sum() / ((Xd - Xd.mean(0)) ** 2).sum()), float((f > 0).float().sum(1).mean())


t = time.time()
mine = train_branch_prism(X)
lost_mine, on_mine = fit(mine, Xt)
print(f"your prism trained in {time.time() - t:.0f} s: on test places {on_mine:.1f} units on, {100 * lost_mine:.1f}% of the variance lost")

Now the lecture's own prism, trained on the full recipe (and, for later, the two it trained with penalty 0.3 and 3). It
is a 1.7 MB file published beside this notebook. On the same test places the two prisms behave alike: a few dozen units
on, a few percent of the variance lost.

In [ ]:
Z = np.load(fetch("https://glass-box-sigma.vercel.app/lectures/17-the-prism/notebook/prisms_l17.npz", "prisms_l17.npz", "the lecture's three trained prisms"))
lecture = {k: {n: torch.tensor(Z[f"{k}_{n}"], device=DEV) for n in ("We", "be", "Wd", "bd")} | {"scale": float(Z[f"{k}_scale"])}
           for k in ("lo", "mid", "hi")}
P999 = {k: Z[f"{k}_p999"] for k in ("lo", "mid", "hi")}        # each unit's 99.9th percentile on photo places
FMAX = {k: Z[f"{k}_fmax"] for k in ("lo", "mid", "hi")}        # each unit's largest value on photo places
prism = lecture["mid"]                                         # penalty 1, the lecture's main prism
lost_lec, on_lec = fit(prism, Xt)
print(f"the lecture's prism on your test places: {on_lec:.1f} units on, {100 * lost_lec:.1f}% lost")
print(f"yours:                                     {on_mine:.1f} units on, {100 * lost_mine:.1f}% lost")
check("your prism and the lecture's: units on agree (ratio)", on_mine / on_lec, 1.0, tol=0.35)
check("your prism and the lecture's: variance lost agrees (ratio)", lost_mine / lost_lec, 1.0, tol=0.5)

### 8. One place, through the prism (Plates XII–XIII and the film)
At the clock's place (3, 14), which of the 768 units turn on, and how well do they rebuild the 96 numbers? Then the same
for the kitchen scale at its place (15, 8). Neuron 3b:359 says about the same on both. Watch units **406** and **640**.

In [ ]:
def at_place(i, r, c):
    a = B5[i, :, r, c]
    f = encode(prism, a).cpu()
    a_hat = rebuild(prism, f.to(DEV)).cpu()
    return a, f, float(((a - a_hat) ** 2).sum() / (a ** 2).sum())


for (i, (r, c), name) in ((0, (3, 14), "clock"), (1, (15, 8), "scale")):
    a, f, err = at_place(i, r, c)
    top = f.argsort(descending=True)[:4].tolist()
    print(f"{name}: neuron 359 = {a[39]:.0f}; {(f > 0).sum().item()} of 768 units on; rebuild loses {100 * err:.1f}%; "
          f"strongest units: " + ", ".join(f"{j} ({f[j]:.0f})" for j in top) + f"; unit 406 = {f[406]:.0f}, unit 640 = {f[640]:.0f}")
_, fc, errc = at_place(0, 3, 14)
_, fs, _ = at_place(1, 15, 8)
check("clock: units on", int((fc > 0).sum()), 56)
check("clock: rebuild loses (share of the 96 numbers' squared length)", errc, 0.0145, tol=0.002)
check("clock: unit 406", fc[406].item(), 653, tol=2)
check("clock: unit 640", fc[640].item(), 0, tol=0.5)
check("clock: unit 421 (a copy of 359, both jobs)", fc[421].item(), 495, tol=2)
check("scale: neuron 359 at (15, 8)", B5[1, 39, 15, 8].item(), 949, tol=1.5)
check("scale: unit 640", fs[640].item(), 393, tol=2)
check("scale: unit 406", fs[406].item(), 0, tol=0.5)

### 9. Ten neurons and their gaps (Plate XIV)
Lecture 10's test: a thin dark arc on white, placed where the middle place (14, 14) of mixed3b looks, turned all the way
round in 2° steps, at seven radii. A neuron's **tuning curve** is what it says at each turn (at its best radius); its
**width** is the run of turns where it says more than half its best; its **bearing** is the way the arc bulges,
clockwise from up. First, find where the middle place looks: the centre of the gradient blob.

In [ ]:
from PIL import ImageDraw

g = torch.full((1, 3, 224, 224), 0.5, device=DEV, requires_grad=True)
with Taps(["mixed3b_5x5_pre_relu_conv"]) as T:
    net(prep(g))
    T["mixed3b_5x5_pre_relu_conv"][0, :, 14, 14].abs().sum().backward()
G = g.grad[0].abs().sum(0).cpu().numpy()
yy, xx = np.mgrid[0:224, 0:224]
CX, CY = float((xx * G).sum() / G.sum()), float((yy * G).sum() / G.sum())
check("receptive-field centre, x", CX, 112.57, tol=0.01)
check("receptive-field centre, y", CY, 111.87, tol=0.01)


def arc(phi, radius, width=2.0, size=224, ss=4):
    # Lecture 10's arc: 2 px wide, drawn at 4x and shrunk; its midpoint on the centre; it bulges toward phi
    # (0 = right, 90 = up); length 1.6 x the radius, at most 90 px
    a = math.radians(phi)
    im = Image.new("L", (size * ss, size * ss), 255)
    cx, cy = CX * ss, CY * ss
    ox, oy = cx - radius * ss * math.cos(a), cy + radius * ss * math.sin(a)
    half = (min(1.6 * radius, 90.0) / radius) / 2
    pts = [(ox + radius * ss * math.cos(a + t), oy - radius * ss * math.sin(a + t)) for t in np.linspace(-half, half, 200)]
    ImageDraw.Draw(im).line(pts, fill=0, width=max(1, int(round(width * ss))), joint="curve")
    im = im.resize((size, size), Image.LANCZOS)
    return torch.from_numpy(np.asarray(im, dtype=np.float32) / 255.0)[None].repeat(3, 1, 1)


RADII = [16, 24, 32, 48, 64, 96, 160]
ANGLES = list(range(0, 360, 2))
t = time.time()
resp = []
with torch.no_grad(), Taps(["mixed3b"]) as T:
    for r in RADII:
        for s in range(0, 180, 90):
            net(prep(torch.stack([arc(a, r) for a in ANGLES[s:s + 90]]).to(DEV)))
            resp.append(T["mixed3b"][:, :, 14, 14].float().cpu())
ARC = torch.cat(resp).reshape(len(RADII), 180, 480).numpy()
BEAR = np.array([((90 - 2 * j) % 360) // 2 for j in range(180)])   # reorder so index k means bearing 2k
NB = ARC[:, BEAR, :]
print(f"{len(RADII) * 180} arcs in {time.time() - t:.0f} s")


def tuning(T):
    # T: radii x 180 (bearing order) -> best radius, its curve, bearing, width (contiguous run above half), opposite
    r = int(np.argmax(T.max(1))); t = T[r]; pk = int(np.argmax(t))
    if t[pk] <= 0:
        return dict(r=r, curve=t, bearing=2 * pk, width=0, opp=1.0, peak=0.0)
    w = 1
    for sgn in (1, -1):
        for d in range(1, 90):
            if t[(pk + sgn * d) % 180] > t[pk] / 2:
                w += 1
            else:
                break
    return dict(r=r, curve=t, bearing=2 * pk, width=2 * w, opp=float(t[(pk + 90) % 180] / t[pk]), peak=float(t[pk]))


def lobes(curve, thr=0.5):
    # the stretches of bearing where a curve is above half its peak: [(from, to), ...] in degrees
    on = curve / curve.max() >= thr
    start = int(np.argmin(on)); runs, cur = [], None
    for k in range(180):
        i = (start + k) % 180
        if on[i] and cur is None:
            cur = i
        if not on[i] and cur is not None:
            runs.append([2 * cur, 2 * ((i - 1) % 180)]); cur = None
    if cur is not None:
        runs.append([2 * cur, 2 * ((start - 1) % 180)])
    return sorted(runs)


TEN = [385, 343, 342, 388, 324, 340, 330, 349, 379, 406]
LECTURE_TEN = {'385': [14, 38], '343': [82, 38], '342': [120, 50], '388': [170, 34], '324': [186, 22], '340': [212, 38], '330': [248, 46], '349': [272, 36], '379': [336, 42], '406': [356, 22], '359': [294, 50]}
ten = {n: tuning(NB[:, :, n]) for n in TEN}
for n in TEN:
    print(f"3b:{n}: bearing {ten[n]['bearing']:3d}°, width {ten[n]['width']}°")
    check(f"3b:{n}: bearing", ten[n]["bearing"], LECTURE_TEN[str(n)][0])
    check(f"3b:{n}: width", ten[n]["width"], LECTURE_TEN[str(n)][1])
gaps_ = sorted(ten[n]["bearing"] for n in TEN)
gap_list = sorted(((b - a) % 360, a, b) for a, b in zip(gaps_, gaps_[1:] + gaps_[:1]))[::-1][:2]
print("the two widest gaps between neighbours:", [f"{a}° to {b}° ({w}°)" for w, a, b in gap_list])

### 10. The missing curves, and where they were (Plates XV–XVI)
Read the prism's units off the very same arcs (the prism takes the 96 branch numbers). A unit counts as a **curve unit**
if the arcs drive it to at least half of its usual strong value on photos (its 99.9th percentile, which the lecture
measured on 400,000 photo places and ships with the prism), its width is 10–120°, and it is silent (under a quarter of its
peak) at the opposite bearing. A **gap unit** is a curve unit at least 16° from every one of the ten.

Then the twist: neuron **3b:359** answers curves bulging two ways. Are the gap units inside its two lobes?

In [ ]:
def circ(a, b):
    return abs((a - b + 180) % 360 - 180)


FB = encode(prism, torch.tensor(NB[:, :, 320:416])).cpu().numpy()            # radii x 180 x 768, bearing order
curve_units = {}
for j in range(768):
    s = tuning(FB[:, :, j])
    if FB[:, :, j].max() <= 0 or P999["mid"][j] <= 0 or FB[:, :, j].max() < 0.5 * P999["mid"][j] or not 10 <= s["width"] <= 120 or s["opp"] >= 0.25:
        continue
    curve_units[j] = s
gap = {j: s for j, s in curve_units.items() if min(circ(s["bearing"], ten[n]["bearing"]) for n in TEN) >= 16}
n359 = tuning(NB[:, :, 359])
L359 = lobes(n359["curve"])
inside = lambda b: any(lo <= b <= hi for lo, hi in L359)
Wp = np.clip(prism["Wd"].cpu().numpy(), 0, None) ** 2                        # each arrow's positive squared parts
built359 = [j for j in gap if int(np.argmax(Wp[:, j])) == 39]
print(f"{len(curve_units)} curve units; {len(gap)} in the gaps; neuron 359's lobes: {L359}")
print(f"{sum(inside(s['bearing']) for s in gap.values())} of the {len(gap)} gap units sit inside 359's lobes; "
      f"{len(built359)} are built mostly from 359")
check("curve units", len(curve_units), 231, tol=2)
check("gap units", len(gap), 56, tol=2)
check("neuron 359's two lobes (degrees)", L359, [[28, 74], [280, 328]])
check("gap units inside 359's lobes", sum(inside(s["bearing"]) for s in gap.values()), 42, tol=2)
check("gap units built mostly from 359", len(built359), 19, tol=1)

fig = plt.figure(figsize=(5.2, 5.2)); ax = fig.add_subplot(projection="polar")
ax.set_theta_zero_location("N"); ax.set_theta_direction(-1)
th = np.radians(np.arange(0, 360, 2))
for n in TEN:
    c = ten[n]["curve"] / ten[n]["peak"]
    ax.fill(th, c, color=INK, alpha=0.18, lw=0); ax.plot(np.append(th, th[0]), np.append(c, c[0]), color=INK, lw=0.8)
c = n359["curve"] / n359["peak"]
ax.plot(np.append(th, th[0]), np.append(c, c[0]), color=INK, lw=2.2, ls="--")
for j in built359:
    c = gap[j]["curve"] / gap[j]["peak"]; ax.plot(np.append(th, th[0]), np.append(c, c[0]), color=INK2, lw=0.8, ls=":")
ax.set_yticks([]); ax.set_title("the ten (shaded), neuron 359 (dashed), gap units built from 359 (dotted)", fontsize=9, pad=14)
plt.show()

### 11. Two jobs, two units (Plates XVII–XVIII)
Unit 406 and unit 640 each answer one of 359's lobes. Draw them on the wheel, then check the lecture's split: on the
clock, 406 fires and 640 is silent; on the scale, the other way round.

In [ ]:
for j, exp in ((406, [54, 48, [359, 415], 0.8475]), (640, [296, 46, [359, 354], 0.775])):
    s = curve_units[j]
    print(f"unit {j}: bearing {s['bearing']}°, width {s['width']}°")
    check(f"unit {j}: bearing", s["bearing"], exp[0], tol=2)
    check(f"unit {j}: width", s["width"], exp[1], tol=2)
fig = plt.figure(figsize=(4.6, 4.6)); ax = fig.add_subplot(projection="polar")
ax.set_theta_zero_location("N"); ax.set_theta_direction(-1)
c = n359["curve"] / n359["peak"]; ax.plot(np.append(th, th[0]), np.append(c, c[0]), color=INK, lw=1.4, ls="--", label="neuron 359")
for j, ls in ((406, "-"), (640, "-")):
    c = curve_units[j]["curve"] / curve_units[j]["peak"]
    ax.fill(th, c, color=INK, alpha=0.25, lw=0); ax.plot(np.append(th, th[0]), np.append(c, c[0]), color=INK, lw=1.6)
    ax.text(np.radians(curve_units[j]["bearing"]), 1.12, f"unit {j}", ha="center", fontsize=9)
ax.set_yticks([]); ax.set_title("one neuron with two jobs; one unit for each", fontsize=10, pad=16); plt.show()

### 12. Turn up the charge (Plate XIX)
Gorton showed what the penalty does to a two-job neuron. The lecture trained three prisms, with penalty 0.3, 1 and 3.
For each, take the units whose arrow leans on 359 (its 359 entry above a quarter of the arrow's largest), and sort them by
the lobes of 359 they answer: **both** (a copy of 359), only the **up-right** lobe, or only the **up-left** one. The
strongest copy's largest value on photos shrinks as the penalty grows, until there is no copy left.

In [ ]:
lob = n359["curve"] / n359["peak"]
L2 = L359[:2]
PEN = {'0.3': {'counts': {'two': 2, 'A': 7, 'B': 10, 'other': 1}, 'two': 747.8}, '1.0': {'counts': {'two': 1, 'A': 11, 'B': 15, 'other': 1}, 'two': 412.7}, '3.0': {'counts': {'two': 0, 'A': 16, 'B': 20, 'other': 0}, 'two': None}}
for k, lam in (("lo", "0.3"), ("mid", "1.0"), ("hi", "3.0")):
    p = lecture[k]
    Wd = p["Wd"].cpu().numpy()
    Fk = encode(p, torch.tensor(NB[:, :, 320:416])).cpu().numpy()[:, :, :]
    kinds = {"two": [], "A": [], "B": [], "other": []}
    for j in range(768):
        if Wd[39, j] <= 0.25 * Wd[39].max() or Fk[:, :, j].max() <= 0:
            continue
        s = tuning(Fk[:, :, j]); c = s["curve"] / max(s["peak"], 1e-9)
        hits = [float(max(c[a // 2:b // 2 + 1].max(), 0)) for a, b in L2]
        kind = "two" if min(hits) >= 0.5 else ("A" if hits[0] >= 0.5 else ("B" if hits[1] >= 0.5 else "other"))
        kinds[kind].append(j)
    two = max(kinds["two"], key=lambda j: FMAX[k][j], default=None)
    print(f"penalty {lam}: both lobes {len(kinds['two'])}, up-right only {len(kinds['A'])}, up-left only {len(kinds['B'])}; "
          f"strongest copy of 359: " + (f"unit {two}, largest value on photos {FMAX[k][two]:.0f}" if two is not None else "none"))
    check(f"penalty {lam}: units copying both lobes", len(kinds["two"]), PEN[lam]["counts"]["two"])
    check(f"penalty {lam}: units on the up-right lobe only", len(kinds["A"]), PEN[lam]["counts"]["A"], tol=1)
    check(f"penalty {lam}: units on the up-left lobe only", len(kinds["B"]), PEN[lam]["counts"]["B"], tol=1)
    if PEN[lam]["two"] is not None:
        check(f"penalty {lam}: the strongest copy's largest value", float(FMAX[k][two]), PEN[lam]["two"], tol=1)

**Did your own prism split 359 too?** Sort your prism's units the same way.

In [ ]:
def kinds_of(p):
    Wd = p["Wd"].cpu().numpy()
    Fk = encode(p, torch.tensor(NB[:, :, 320:416])).cpu().numpy()
    out = {"two": [], "A": [], "B": [], "other": []}
    for j in range(Wd.shape[1]):
        if Wd[39, j] <= 0.25 * Wd[39].max() or Fk[:, :, j].max() <= 0:
            continue
        s = tuning(Fk[:, :, j]); c = s["curve"] / max(s["peak"], 1e-9)
        hits = [float(max(c[a // 2:b // 2 + 1].max(), 0)) for a, b in L2]
        out["two" if min(hits) >= 0.5 else ("A" if hits[0] >= 0.5 else ("B" if hits[1] >= 0.5 else "other"))].append(j)
    return out


mk = kinds_of(mine)
print(f"your prism (penalty 1): both lobes {len(mk['two'])}, up-right only {len(mk['A'])}, up-left only {len(mk['B'])}")
cosines = (F.normalize(mine["Wd"], dim=0).T @ F.normalize(prism["Wd"], dim=0)).max(0).values.cpu()
print(f"{int((cosines > 0.9).sum())} of the lecture's 768 units have a near twin in yours (arrows within cos 0.9)")
check("your prism has units for each of 359's two jobs", len(mk["A"]) > 0 and len(mk["B"]) > 0, True)

### 13. What the new curves are made of, and how sharp (Plates XX–XXI)
Write a unit's arrow out in the 96 neurons (Lecture 14: an arrow can mix neurons). The share of its squared length on a
set of neurons says what it is made of. The *curve-selective* neurons are the 40 the lecture found answer drawn curves
strongly (it measured their usual strength on 400,000 photo places; the list is copied here).

Then sharpness: for each of the ten detectors, the curve unit whose tuning (over every radius and bearing) correlates
best with it. Goodfire's 2026 essay saw blurry prism curves; on the real InceptionV1 they are as sharp as the neurons.

In [ ]:
CURVE_SELECTIVE = [322, 324, 327, 330, 334, 335, 337, 338, 339, 340, 342, 343, 344, 346, 348, 349, 352, 357, 359, 360, 362, 365, 367, 373, 376, 379, 380, 381, 384, 385, 389, 394, 395, 402, 406, 407, 408, 409, 411, 413]
share = lambda j, idx: float(Wp[idx, j].sum() / Wp[:, j].sum())
top2 = lambda j: [320 + int(i) for i in np.argsort(-Wp[:, j])[:2]]
for j, exp in ((406, [54, 48, [359, 415], 0.8475]),):
    print(f"unit {j}: made mostly of {top2(j)}, which hold {100 * share(j, np.argsort(-Wp[:, j])[:2]):.0f}% of its arrow; "
          f"{100 * share(j, [n - 320 for n in CURVE_SELECTIVE]):.0f}% is in curve-selective neurons, "
          f"{100 * share(j, [n - 320 for n in TEN]):.1f}% in the ten")
    check(f"unit {j}: its top two neurons", top2(j), exp[2])
    check(f"unit {j}: their share", share(j, np.argsort(-Wp[:, j])[:2]), exp[3], tol=0.002)
med = lambda idx: float(np.median([share(j, idx(j)) for j in gap]))
m_ten, m_cs, m_top2 = med(lambda j: [n - 320 for n in TEN]), med(lambda j: [n - 320 for n in CURVE_SELECTIVE]), med(lambda j: np.argsort(-Wp[:, j])[:2])
print(f"all {len(gap)} gap units (medians): {100 * m_ten:.1f}% in the ten, {100 * m_cs:.0f}% in curve-selective neurons, "
      f"{100 * m_top2:.0f}% in their top two")
check("gap units: share in the ten (median)", m_ten, 0.0174, tol=0.003)
check("gap units: share in curve-selective neurons (median)", m_cs, 0.8946, tol=0.01)
check("gap units: share in their top two (median)", m_top2, 0.8347, tol=0.01)

shapes, widths = [], []
for n in TEN:
    v = NB[:, :, n].reshape(-1)
    c, j = max((float(np.corrcoef(v, FB[:, :, j].reshape(-1))[0, 1]), j) for j in curve_units)
    shapes.append(c); widths.append(curve_units[j]["width"])
    print(f"3b:{n} ({ten[n]['width']}° wide): best match unit {j}, {curve_units[j]['width']}° wide, shape match {c:.3f}")
check("the matches' narrowest width", min(widths), 22, tol=2)
check("the matches' widest width", max(widths), 46, tol=2)
check("the worst shape match", min(shapes), 0.9441, tol=0.01)

## Your turn
Each exercise has a hint; worked solutions (with their checks) are at the end.

1. **Keep the strongest one (Plate VIII).** Train five TopK prisms on the toy with `topk=1` (no penalty). How many of the
   five line up with the ideas (line-up above 0.99)? *Hint: `train_prisms(h_train, 5, seeds=range(5), topk=1)`.*
2. **A bigger prism (Plate X).** Train a 20-unit prism on the toy with penalty 0.1 and 12,000 steps. How many units
   ever turn on for the test inputs? *Hint: a unit is alive if `units(p, h_test)[:, j]` is ever above a millionth.*
3. **Unit 640's recipe (Plate XX).** Which two neurons is unit 640 mostly made of, and what share of its arrow do they
   hold? *Hint: reuse `top2` and `share`.*
4. **Where does 640 fire?** Find the place in the scale photo where unit 640 is strongest. *Hint: encode all 784 places
   of `B5[1]` at once.*
5. **Charge your own prism more.** Train your own branch prism with penalty 3. Do fewer units turn on? *Hint:
   `train_branch_prism(X, lam=3.0)`, then `fit(...)` on `Xt`.*

In [ ]:
# 1. TopK, k = 1, five seeds: how many line up?
# TODO

In [ ]:
# 2. A 20-unit prism, penalty 0.1: how many units are alive?
# TODO

In [ ]:
# 3. Unit 640: its top two neurons and their share
# TODO

In [ ]:
# 4. The place where unit 640 is strongest in the scale photo
# TODO

In [ ]:
# 5. Your own prism at penalty 3: units on and variance lost
# TODO

## Solutions

In [ ]:
# 1. Only the lucky seeds find the ideas; the others get stuck, some with units that never turn on (dead units).
topk = train_prisms(h_train, 5, seeds=range(5), topk=1)
good = [lineup(p)[0] > 0.99 for p in topk]
dead = [int((units(p, h_test) > 1e-6).sum(0).eq(0).sum()) for p in topk]     # never above a millionth: dead
rarest = [float((units(p, h_test) > 1e-6).float().mean(0).min()) for p in topk]
print("TopK k = 1: lines up", good, "· dead units per seed", dead, "(the lecture's run:", [1, 3, 0, 3, 0], ")")
print("each seed's least-used unit is on for", [f"{100 * r:.2f}%" for r in rarest], "of the test inputs")
check("TopK k = 1: seeds that line up", sum(good), 1)
# A unit on for a handful of the 50,000 test inputs is dead in one run and barely alive in another (tiny differences in
# arithmetic between machines decide it), so compare the total over the five seeds.
check("TopK k = 1: dead units, five seeds together", sum(dead), 7, tol=1)

In [ ]:
# 2. Twenty units for five ideas: several pile onto one idea, and some never turn on.
wide = train_prisms(h_train, 20, seeds=[0], lam=0.1, steps=12000)[0]
alive = int((units(wide, h_test) > 1e-6).any(0).sum())
print(f"20 units: {alive} alive")
check("20-unit prism, penalty 0.1: alive units", alive, 16)

In [ ]:
# 3. Unit 640, like 406, is mostly 359 plus one neighbour.
print("unit 640:", top2(640), f"{100 * share(640, np.argsort(-Wp[:, 640])[:2]):.1f}%")
check("unit 640: its top two neurons", top2(640), [359, 354])
check("unit 640: their share", share(640, np.argsort(-Wp[:, 640])[:2]), 0.775, tol=0.002)

In [ ]:
# 4. Encode every place of the scale photo and find unit 640's strongest.
f_all = encode(prism, B5[1].flatten(1).T).cpu()                    # 784 places x 768 units
k = int(f_all[:, 640].argmax())
print(f"unit 640 is strongest at place ({k // 28}, {k % 28}): {f_all[k, 640]:.0f}")
check("the scale photo: unit 640's strongest place", [k // 28, k % 28], [15, 8])

In [ ]:
# 5. A harder charge: fewer units on, more of the variance lost (the lecture: 13.6 on, 24.1% lost at penalty 3,
#    against 34.6 on and 5.5% at penalty 1, on its own training places).
mine3 = train_branch_prism(X, lam=3.0)
lost3, on3 = fit(mine3, Xt)
print(f"penalty 3: {on3:.1f} units on, {100 * lost3:.1f}% lost (penalty 1: {on_mine:.1f} on, {100 * lost_mine:.1f}% lost)")
check("penalty 3 turns fewer units on than penalty 1", on3 < on_mine, True)
check("penalty 3 loses more of the variance", lost3 > lost_mine, True)
print(f"whole notebook: {(time.time() - T0) / 60:.1f} minutes")